In [ ]:
SCORES_DIR = './scores'
os.makedirs(SCORES_DIR, exist_ok=True)

for sub in SUBS:
    out = identify_subject(sub, use_weight=True, verbose=(sub in DEV_SUBS))
    np.savez_compressed(f'{SCORES_DIR}/encoding_sub-{sub:02d}.npz',
                        S=out['S'], w=out['w'],
                        test_ids=TEST_IDS.astype(np.int64))
print('wrote', sorted(os.listdir(SCORES_DIR)))

## Export similarity matrices for 06

06 fuses these with the contrastive decoder's scores. We write **one `.npz` per subject** with `S` and the per-(ch,t) weight.

In [ ]:
def summary_table(df):
    g = df.groupby('tag')[['top1', 'top5', 'median_rank']]
    out = pd.DataFrame({
        'top1_mean':   g.top1.mean(),
        'top1_sd':     g.top1.std(),
        'top5_mean':   g.top5.mean(),
        'top5_sd':     g.top5.std(),
        'rank_median': g.median_rank.median(),
        'n':           g.top1.count(),
    }).sort_values('top1_mean', ascending=False)
    return out

summary_table(results)

## Results table (mean ± sd on sub-01..04)

In [ ]:
# +MVNN only (vs baseline), still single bank
def ab_mvnn_only(sub):
    # preprocess_subject already does MVNN; isolate by rerunning with do_mvnn=False and compare
    # we already pay for MVNN in the default, so just rerun baseline but using the full preprocess:
    Y_tst, Y_pred = fit_single_bank(sub, 'alexnet_features.3')
    S = weighted_sim(Y_tst, Y_pred)
    return topk(ranked_ids(S), load_test_labels(sub))

# multi-layer AlexNet via banded ridge, unweighted ident
def ab_alexnet_bands(sub):
    banks = ['alexnet_features.0', 'alexnet_features.3', 'alexnet_features.6',
             'alexnet_features.8', 'alexnet_features.10']
    out = identify_subject(sub, banks=banks, use_weight=False, verbose=False)
    return out['score']

# full bank set, unweighted
def ab_full_banks_unweighted(sub):
    out = identify_subject(sub, use_weight=False, verbose=False)
    return out['score']

# full bank set, weighted by CV predictability
def ab_full_banks_weighted(sub):
    out = identify_subject(sub, use_weight=True, verbose=False)
    return out['score']

eval_tag('alexnet_bands_unweighted', ab_alexnet_bands)
eval_tag('full_banks_unweighted',   ab_full_banks_unweighted)
eval_tag('full_banks_weighted',     ab_full_banks_weighted)

In [ ]:
import pandas as pd

RESULTS_PATH = f'{CACHE_DIR}/results.csv'
if os.path.exists(RESULTS_PATH):
    results = pd.read_csv(RESULTS_PATH)
else:
    results = pd.DataFrame(columns=['tag', 'sub', 'top1', 'top5', 'median_rank'])

def log_row(tag, sub, score):
    global results
    row = {'tag': tag, 'sub': sub, **score}
    results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)
    results.to_csv(RESULTS_PATH, index=False)

def eval_tag(tag, score_fn):
    print(f'=== {tag} ===')
    for s in DEV_SUBS:
        sc = score_fn(s)
        log_row(tag, s, sc)
        print(f'  sub-{s:02d}', sc)
    sub = results[results.tag == tag]
    print('  mean top1={:.3f} top5={:.3f}'.format(sub.top1.mean(), sub.top5.mean()))

# baseline: only AlexNet features.3, unweighted ident, matches 01/02
def ab_baseline(sub):
    Y_tst, Y_pred = fit_single_bank(sub, 'alexnet_features.3')
    S = weighted_sim(Y_tst, Y_pred)
    return topk(ranked_ids(S), load_test_labels(sub))

eval_tag('baseline_alexnet3', ab_baseline)

## Ablations

One cell per quick win. Each writes a row into the running table and does **not** touch sub-01..04 tst labels for anything other than scoring.

In [ ]:
res = {s: identify_subject(s) for s in DEV_SUBS}

In [ ]:
def predictability_weight(Y_true_cv, Y_pred_cv):
    """Per (ch, t) correlation across train trials, floored at 0.
    Y_*_cv: (n, ch, t)."""
    a = Y_true_cv - Y_true_cv.mean(0, keepdims=True)
    b = Y_pred_cv - Y_pred_cv.mean(0, keepdims=True)
    num = (a * b).mean(0)
    den = (a.std(0) * b.std(0) + 1e-6)
    r = num / den
    return np.clip(r, 0, None).astype(np.float32)        # (ch, t)

def identify_subject(sub, banks=DEFAULT_BANKS, use_weight=True,
                     tmin_idx=5, tmax_idx=60, verbose=True):
    Y_tst, Y_pred_test, Y_pred_cv, ids = fit_banded_ridge(sub, banks=banks)
    _, Y_trn, _, _, _ = preprocess_subject(sub)
    w = predictability_weight(Y_trn, Y_pred_cv) if use_weight else None
    S = weighted_sim(Y_tst, Y_pred_test, w=w, tmin_idx=tmin_idx, tmax_idx=tmax_idx)
    out = {'S': S, 'w': w}
    if sub in DEV_SUBS:
        out['score'] = topk(ranked_ids(S), load_test_labels(sub))
        if verbose: print(f'sub-{sub:02d}', out['score'])
    return out

In [ ]:
from himalaya.backend  import set_backend
from himalaya.ridge     import GroupRidgeCV
from himalaya.scoring   import r2_score_split

set_backend('torch_cuda' if GPU else 'numpy')

DEFAULT_BANKS = [
    'lowlevel',
    'alexnet_features.0', 'alexnet_features.3', 'alexnet_features.6',
    'alexnet_features.8', 'alexnet_features.10',
    'clip_vitl14',
    'dinov2_vitl14',
    'videomae_v2',
    'clap',
]

def prepare_banks(sub, banks=DEFAULT_BANKS, var=0.95):
    ids, Y_trn, Y_tst, _, _ = preprocess_subject(sub)
    Xs_trn, Xs_tst, slices = [], [], []
    cur = 0
    for b in banks:
        if not os.path.exists(f'{FEAT_DIR}/{b}.npz'):
            print('skip missing bank', b); continue
        Xt_raw = stack_bank(b, ids)
        Xq_raw = stack_bank(b, TEST_IDS)
        Xt, Xq, _ = fit_bank_pca(Xt_raw, Xq_raw, var=var)
        Xs_trn.append(Xt); Xs_tst.append(Xq)
        slices.append((b, slice(cur, cur + Xt.shape[1])))
        cur += Xt.shape[1]
    X_trn = np.concatenate(Xs_trn, axis=1)
    X_tst = np.concatenate(Xs_tst, axis=1)
    groups = [s for _, s in slices]
    return ids, Y_trn, Y_tst, X_trn, X_tst, groups, slices

def fit_banded_ridge(sub, banks=DEFAULT_BANKS, var=0.95, n_folds=5, cache=True):
    cp = f'{CACHE_DIR}/br_sub-{sub:02d}.npz'
    if cache and os.path.exists(cp):
        z = np.load(cp)
        return z['Y_tst'], z['Y_pred_test'], z['Y_pred_cv'], z['ids']

    ids, Y_trn, Y_tst, X_trn, X_tst, groups, slices = prepare_banks(sub, banks, var)
    n, ch, t = Y_trn.shape
    Yflat = Y_trn.reshape(n, -1)

    # out-of-fold predictions on train, for the per-(ch,t) predictability weight
    Y_pred_cv = np.zeros_like(Yflat)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=0)
    for i, (tr, va) in enumerate(kf.split(X_trn)):
        m = GroupRidgeCV(groups=groups,
                         random_state=0,
                         cv=5,
                         solver_params={'n_targets_batch': 1024})
        m.fit(X_trn[tr], Yflat[tr])
        Y_pred_cv[va] = np.asarray(m.predict(X_trn[va]))
        print(f'fold {i+1}/{n_folds} done')

    # refit on full train for test predictions
    final = GroupRidgeCV(groups=groups, random_state=0, cv=5,
                         solver_params={'n_targets_batch': 1024})
    final.fit(X_trn, Yflat)
    Y_pred_test = np.asarray(final.predict(X_tst)).reshape(len(TEST_IDS), ch, t)
    Y_pred_cv_3d = Y_pred_cv.reshape(n, ch, t)

    if cache:
        np.savez_compressed(cp, ids=ids, Y_tst=Y_tst,
                            Y_pred_test=Y_pred_test, Y_pred_cv=Y_pred_cv_3d)
    return Y_tst, Y_pred_test, Y_pred_cv_3d, ids

## Banded ridge (himalaya)

Each feature bank becomes one "band"; himalaya tunes a separate ridge penalty per band on 5-fold CV of training videos. Then we predict EEG for every candidate video (train+test ∪ test-only shown in notebook 06).

Output of this cell:
- `Y_pred_test`: (102, ch, t) predicted EEG for the 102 candidate videos
- `Y_pred_cv`:   (n_train, ch, t) out-of-fold predicted EEG for training videos — used to build the per-(ch,t) weight.

In [ ]:
def fit_single_bank(sub, bank='alexnet_features.3'):
    ids, Y_trn, Y_tst, _, _ = preprocess_subject(sub)
    X_trn_raw = stack_bank(bank, ids)
    X_tst_raw = stack_bank(bank, TEST_IDS)
    X_trn, X_tst, _ = fit_bank_pca(X_trn_raw, X_tst_raw)

    n, ch, t = Y_trn.shape
    model = RidgeCV(alphas=np.logspace(-3, 6, 20))
    model.fit(X_trn, Y_trn.reshape(n, -1))
    Y_pred = model.predict(X_tst).reshape(len(TEST_IDS), ch, t)
    return Y_tst, Y_pred

Y_tst, Y_pred = fit_single_bank(1, 'alexnet_features.3')
S = weighted_sim(Y_tst, Y_pred)                                   # unweighted
print('baseline sub-01:', topk(ranked_ids(S), load_test_labels(1)))

## Baseline reproducer (sanity check against 01/02)

Single-bank AlexNet `features.3` + PCA(0.95) + RidgeCV on preprocessed-to-100Hz EEG, unweighted identification. Should land near 3.9% top-1 / 11.8% top-5 on sub-01.

In [ ]:
def weighted_sim(Y_true, Y_pred, w=None, tmin_idx=5, tmax_idx=60):
    """Y_true: (n_test, ch, t), Y_pred: (n_vid, ch, t), w: (ch, t) or None.
    Returns S of shape (n_test, n_vid) — weighted dot product on z-scored flat vectors."""
    sl = slice(tmin_idx, tmax_idx)
    A = Y_true[:, :, sl]
    B = Y_pred[:, :, sl]
    if w is None:
        W = np.ones(A.shape[1:], dtype=np.float32)
    else:
        W = np.maximum(w[:, sl], 0).astype(np.float32)
    Wf = W.reshape(-1)
    Af = A.reshape(A.shape[0], -1) * Wf
    Bf = B.reshape(B.shape[0], -1) * Wf
    Af = (Af - Af.mean(1, keepdims=True)) / (Af.std(1, keepdims=True) + 1e-6)
    Bf = (Bf - Bf.mean(1, keepdims=True)) / (Bf.std(1, keepdims=True) + 1e-6)
    return Af @ Bf.T / Af.shape[1]

## Identification scoring

Convention from 00: `S[r, j]` = similarity between recording `r` (in original tst order) and candidate video `TEST_IDS[j]`. We rank videos per recording and pick top-k.

We use a **weighted correlation** between the predicted EEG of each candidate video and the measured EEG of each test recording, with per-(channel×time) weights set by the train-CV R² of the encoder. Low-predictability samples pull less on the identification score.

In [ ]:
def load_bank(name):
    z = np.load(f'{FEAT_DIR}/{name}.npz')
    return z, lambda vid: z[f'{int(vid):04d}']

def stack_bank(name, vids):
    _, get = load_bank(name)
    return np.stack([get(v) for v in vids]).astype(np.float32)

def fit_bank_pca(X_trn, X_tst, var=0.95):
    sc  = StandardScaler().fit(X_trn)
    Xtz = sc.transform(X_trn)
    pca = PCA(n_components=var, svd_solver='full').fit(Xtz)
    Xtp = pca.transform(Xtz)
    sc2 = StandardScaler().fit(Xtp)
    Xts = sc2.transform(Xtp)
    Xq  = sc2.transform(pca.transform(sc.transform(X_tst)))
    return Xts.astype(np.float32), Xq.astype(np.float32), (sc, pca, sc2)

def available_banks():
    return sorted(os.path.basename(p)[:-4] for p in glob.glob(f'{FEAT_DIR}/*.npz'))

print('banks:', available_banks())

## Feature banks

03 wrote one `.npz` per bank, keyed by 4-digit video id (`"0001".."1102"`). We align each bank to a given list of video ids, then z-score + PCA on train.

In [ ]:
def preprocess_subject(sub, do_mvnn=True, cache=True):
    """Return (trn_ids, Y_trn, Y_tst, grand_erp, W). Caches per subject."""
    cp = f'{CACHE_DIR}/pp_sub-{sub:02d}.npz'
    if cache and os.path.exists(cp):
        z = np.load(cp)
        return z['ids'], z['Y_trn'], z['Y_tst'], z['erp'], z['W']

    ids, eeg_trn = load_train(sub)                     # (n, 17, 1500)
    eeg_tst      = load_test(sub)                      # (102, 17, 1500)

    # band-pass + decimate at 500 Hz, then down to 100 Hz
    eeg_trn = decimate_to_100(bandpass(eeg_trn))       # (n, 17, 300)
    eeg_tst = decimate_to_100(bandpass(eeg_tst))

    # train-only grand ERP subtraction (keeps stimulus-specific variance)
    erp = eeg_trn.mean(0, keepdims=True)
    eeg_trn = eeg_trn - erp
    eeg_tst = eeg_tst - erp

    # z-score each split by its own mean/std (test std ~0.65x train, per plan)
    eeg_trn, mu_t, sd_t = zscore_split(eeg_trn)
    eeg_tst, _, _       = zscore_split(eeg_tst)

    if do_mvnn:
        eeg_trn, eeg_tst, W = mvnn(eeg_trn, eeg_tst)
    else:
        W = np.eye(eeg_trn.shape[1], dtype=np.float32)

    if cache:
        np.savez_compressed(cp, ids=ids, Y_trn=eeg_trn, Y_tst=eeg_tst, erp=erp, W=W)
    return ids, eeg_trn, eeg_tst, erp, W

ids1, Y_trn1, Y_tst1, erp1, W1 = preprocess_subject(1)
print(Y_trn1.shape, Y_tst1.shape, 'erp:', erp1.shape, 'W:', W1.shape)

In [ ]:
import mne

def bandpass(x, sfreq=SFREQ, l=0.1, h=40.):
    """Zero-phase band-pass. x: (n, ch, t) -> same shape."""
    flat = x.reshape(-1, x.shape[-1])
    y = mne.filter.filter_data(flat.astype(np.float64), sfreq=sfreq,
                               l_freq=l, h_freq=h, verbose=False)
    return y.reshape(x.shape).astype(np.float32)

def decimate_to_100(x, dec=DEC):
    return decimate(x, dec, axis=-1, ftype='fir', zero_phase=True).astype(np.float32)

def mvnn(x_train, x_test):
    """Multivariate noise normalisation: whiten channels by train residual covariance.
    x_*: (n, ch, t). Returns the two splits in the whitened space."""
    r = x_train - x_train.mean(0, keepdims=True)                   # (n, ch, t)
    r = r.transpose(0, 2, 1).reshape(-1, x_train.shape[1])         # (n*t, ch)
    cov = np.cov(r, rowvar=False) + 1e-6 * np.eye(r.shape[1])
    # ZCA whitening
    u, s, vt = np.linalg.svd(cov)
    W = u @ np.diag(1 / np.sqrt(s)) @ u.T
    def apply(a):
        shp = a.shape
        flat = a.transpose(0, 2, 1).reshape(-1, shp[1]) @ W.T
        return flat.reshape(shp[0], shp[2], shp[1]).transpose(0, 2, 1).astype(np.float32)
    return apply(x_train), apply(x_test), W

def zscore_split(x, mu=None, sd=None):
    """z-score each (ch, t) feature across the trials of one split."""
    if mu is None:
        mu = x.mean(0, keepdims=True)
        sd = x.std(0, keepdims=True) + 1e-6
    return (x - mu) / sd, mu, sd

## EEG preprocessing

Done once per subject on train+test (filters are causal-free zero-phase so no leakage), then split-specific steps (z-score, grand-ERP subtraction, MVNN) are applied per split. We keep the final shape `(n, 17, 300)` on a 100 Hz grid.

In [ ]:
TRAIN_DIR     = './trn'
TEST_DIR      = './tst'
TEST_STIM_DIR = './tst_stim'
FEAT_DIR      = './features'
CACHE_DIR     = './cache_04'
os.makedirs(CACHE_DIR, exist_ok=True)

SFREQ         = 500
DEC           = 5                        # -> 100 Hz
SFREQ_D       = SFREQ // DEC
SUBS          = [1, 2, 3, 4, 5, 6]
DEV_SUBS      = [1, 2, 3, 4]
HELDOUT_SUBS  = [5, 6]
TEST_IDS      = np.arange(1001, 1103)    # 102 candidate videos

def load_train(sub):
    z = np.load(f'{TRAIN_DIR}/sub-{sub:02d}_trn.npz')
    ids = np.array(sorted(z.files, key=int), dtype=int)
    eeg = np.stack([z[str(i)] for i in ids])
    return ids, eeg

def load_test(sub):
    return np.load(f'{TEST_DIR}/sub-{sub:02d}_tst.npy')

def load_test_labels(sub):
    assert sub in DEV_SUBS, f'test labels for sub-{sub:02d} are withheld'
    return np.load(f'{TEST_STIM_DIR}/sub-{sub:02d}_tst_stim.npy').astype(int)

def ranked_ids(S, video_ids=TEST_IDS):
    return np.asarray(video_ids)[np.argsort(-S, axis=1)]

def topk(ranked, truth, ks=(1, 5)):
    hits = ranked == np.asarray(truth)[:, None]
    assert hits.any(1).all(), 'some true videos are missing from the ranking'
    rank = hits.argmax(1) + 1
    out = {f'top{k}': float((rank <= k).mean()) for k in ks}
    out['median_rank'] = float(np.median(rank))
    return out

## Config and helpers (duplicated from 00 so this notebook runs standalone)

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import decimate
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold

try:
    import torch
    GPU = torch.cuda.is_available()
except Exception:
    GPU = False
print('gpu:', GPU)

In [ ]:
# %pip install -r requirements.txt      # local
# Kaggle: himalaya and mne are the only new things needed here if 03 already ran.

# 04 — Encoding model (per subject)

Per-subject video→EEG encoding model, then video identification by nearest neighbour in predicted-EEG space. Produces similarity matrices `S[sub]` of shape `(n_test, 102)` that notebook 06 fuses with the contrastive decoder's scores.

Pipeline per subject:
1. Load EEG train (sorted by video id) and test (original shuffled order).
2. Preprocess EEG: band-pass 0.1–40 Hz, decimate ×5 (100 Hz), per-split z-score, train-only grand-ERP subtraction, MVNN (multivariate noise normalisation).
3. Stack feature banks from 03 (`alexnet_*`, `clip`, `videomae_v2`, `clap`, `lowlevel`) and per-bank PCA.
4. Banded ridge (himalaya) video→EEG, 5-fold CV over training videos, per channel×time.
5. Build the identification similarity: predicted EEG for every candidate video vs. measured test EEG, weighted by per-(channel,time) CV predictability.
6. Score top-1/top-5 on sub-01..04 test, cache `S[sub]`.

Rules:
- `tst` of sub-01..04 is used **only** for scoring. All model selection (alphas, feature banks, weightings) uses CV over train.
- Submission rows for sub-05/06 stay in `load_test(sub)` order — never reorder here.

Baseline sanity: cell "Baseline reproducer" at the bottom should recover ≈3.9% top-1 / ≈11.8% top-5 on sub-01 using AlexNet `features.3` + PCA(0.95) + RidgeCV, matching notebooks 01/02.